# 딥러닝및실습-2차-이성수-2019136093

In [18]:
!pip install torch torchvision torchaudio
!pip install wandb
!pip install pandas
!pip install scikit-learn

INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 4.8 MB/s  0:00:00 eta 0:00:01


# [요구사항 1] titanic 딥러닝 모델 기본 훈련

1. _01_code/_08_learning_and_optimization/c_my_model_training_with_argparse_wandb.py코드를 그대로 활용하되 titanic 데이터에 맞게 수정하여 코딩하기
2. Wandb로 훈련 과정 데이터 올려 그래프 얻어 내기
    - • Training loss• Validation loss• 위 두 그래프를 보여주는 Wandb URL 얻어내기
    -  Wandb 페이지 생성 PDF 인쇄
        1. x축 Epoch에 대하여, y축에 Training loss 변화를 보여주는 그래프 제시
        2. x축 Epoch에 대하여, y축에 Validation loss 변화를 보여주는 그래프 제시
        3. 위 두 그래프를 포함하고 있는 Wandb 제출페이지를 PDF 인쇄

In [23]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""Titanic 과제 – 데이터 로드 → 모델 → 학습 → W&B
   경로 설정, 자동 __init__.py 생성, 그리고 전체 파이프라인을 한 파일에 정리합니다.
"""

# -------------------------------------------------
# 1️⃣ 현재 파일(스크립트 혹은 Notebook) 경로 찾기
# -------------------------------------------------
import sys
from pathlib import Path
import inspect

# Notebook / IPython 환경이면 inspect 로, 그렇지 않으면 __file__ 로
try:
    # Jupyter / IPython
    CURRENT_FILE = Path(inspect.getfile(inspect.currentframe())).resolve()
except Exception:                     # __file__ 가 없을 때 (예: .py 실행)
    CURRENT_FILE = Path(__file__).resolve()

# 프로젝트 루트는 현재 파일 기준으로 **두 단계 위** (…/link_dl)
BASE_PATH = str(CURRENT_FILE.parents[2])
print("🔧 BASE_PATH =", BASE_PATH)

# -------------------------------------------------
# 2️⃣ 루트를 sys.path 에 추가 (이미 있으면 건너뛰기)
# -------------------------------------------------
if BASE_PATH not in sys.path:
    sys.path.append(BASE_PATH)

# -------------------------------------------------
# 3️⃣ __init__.py 가 없으면 자동 생성 (한 번만 실행하면 OK)
# -------------------------------------------------
import os
PKG_DIRS = [
    "_01_code",
    "_01_code/_03_real_world_data_to_tensors",
    "_03_homeworks",
    "_03_homeworks/homework_2",
]
for d in PKG_DIRS:
    init_path = os.path.join(BASE_PATH, d, "__init__.py")
    if not os.path.isfile(init_path):
        open(init_path, "a").close()   # 빈 파일 만들기

# -------------------------------------------------
# 4️⃣ 이제 정상적인 import 사용
# -------------------------------------------------
# (예시) CaliforniaHousingDataset
from _01_code._03_real_world_data_to_tensors.m_california_housing_dataset_dataloader import (
    CaliforniaHousingDataset,
)

# (예시) Titanic 전처리·Dataset
from _03_homeworks.homework_2.titanic_dataset import get_preprocessed_dataset

# -------------------------------------------------
# 5️⃣ 기본 라이브러리 및 W&B 설정
# -------------------------------------------------
import argparse
from datetime import datetime

import wandb
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, random_split

# -------------------------------------------------
# 6️⃣ Titanic 전용 DataLoader
# -------------------------------------------------
def get_data():
    """
    Titanic 전처리 데이터를 로드하고,
    wandb.config.batch_size 로 DataLoader 를 만든 뒤 반환합니다.
    """
    train_ds, val_ds, _ = get_preprocessed_dataset()

    train_loader = DataLoader(
        dataset=train_ds,
        batch_size=wandb.config.batch_size,
        shuffle=True,
        drop_last=False,
    )
    val_loader = DataLoader(
        dataset=val_ds,
        batch_size=len(val_ds),   # 전체 검증을 한 번에
        shuffle=False,
    )
    return train_loader, val_loader


# -------------------------------------------------
# 7️⃣ 모델 정의 (입력 차원 10, 출력 1)
# -------------------------------------------------
class MyModel(nn.Module):
    def __init__(self, n_input: int, n_output: int):
        super().__init__()

        # ---------- 활성화 함수 선택 ----------
        act_name = wandb.config.get("activation", "relu").lower()
        act_map = {
            "relu": nn.ReLU,
            "sigmoid": nn.Sigmoid,
            "elu": nn.ELU,
            "leaky_relu": nn.LeakyReLU,
        }
        activation = act_map.get(act_name, nn.ReLU)   # 기본 ReLU

        # ---------- 네트워크 ----------
        self.net = nn.Sequential(
            nn.Linear(n_input, wandb.config.n_hidden_unit_list[0]),
            activation(),
            nn.Linear(wandb.config.n_hidden_unit_list[0],
                      wandb.config.n_hidden_unit_list[1]),
            activation(),
            nn.Linear(wandb.config.n_hidden_unit_list[1], n_output),
        )

    def forward(self, x):
        return self.net(x)


def get_model_and_optimizer():
    """
    Titanic 데이터는 10개의 피처 → n_input=10,
    이진 분류 → n_output=1
    """
    model = MyModel(n_input=10, n_output=1)
    optimizer = optim.SGD(model.parameters(),
                          lr=wandb.config.learning_rate)
    return model, optimizer


# -------------------------------------------------
# 8️⃣ 학습 루프
# -------------------------------------------------
def training_loop(model, optimizer, train_loader, val_loader):
    n_epochs = wandb.config.epochs
    loss_fn = nn.BCEWithLogitsLoss()   # 이진 분류에 적합
    next_print = 100

    for epoch in range(1, n_epochs + 1):
        # ----------- Train -----------
        model.train()
        train_loss_sum = 0.0
        train_steps = 0
        for batch in train_loader:
            inputs = batch["input"]
            targets = batch["target"].float().unsqueeze(1)   # (B,1)

            outputs = model(inputs)
            loss = loss_fn(outputs, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item()
            train_steps += 1

        # ----------- Validation -----------
        model.eval()
        val_loss_sum = 0.0
        val_steps = 0
        with torch.no_grad():
            for batch in val_loader:
                inputs = batch["input"]
                targets = batch["target"].float().unsqueeze(1)

                outputs = model(inputs)
                loss = loss_fn(outputs, targets)

                val_loss_sum += loss.item()
                val_steps += 1

        # ----------- Logging ----------
        wandb.log(
            {
                "epoch": epoch,
                "train_loss": train_loss_sum / train_steps,
                "val_loss":   val_loss_sum / val_steps,
            }
        )

        if epoch >= next_print:
            print(
                f"Epoch {epoch:4d} | "
                f"Train loss {train_loss_sum/train_steps:.4f} | "
                f"Val loss {val_loss_sum/val_steps:.4f}"
            )
            next_print += 100


# -------------------------------------------------
# 9️⃣ 메인 엔트리포인트
# -------------------------------------------------
def main(args):
    run_id = datetime.now().astimezone().strftime("%Y-%m-%d_%H-%M-%S")

    # ---- wandb 설정 ----
    config = {
        "epochs": args.epochs,
        "batch_size": args.batch_size,
        "learning_rate": 1e-3,
        "n_hidden_unit_list": [20, 20],
        "activation": args.activation,   # 커맨드라인에서 지정
    }

    wandb.init(
        mode="online" if args.wandb else "disabled",
        project="titanic_homework2",
        notes="Titanic assignment – activation / batch‑size study",
        tags=["titanic", "homework2"],
        name=run_id,
        config=config,
    )

    # ---- 데이터 로드 ----
    train_loader, val_loader = get_data()

    # ---- 모델 / 옵티마이저 ----
    model, optimizer = get_model_and_optimizer()

    print("#" * 50, "START TRAINING")

    # ---- 학습 루프 ----
    training_loop(
        model=model,
        optimizer=optimizer,
        train_loader=train_loader,
        val_loader=val_loader,
    )

    wandb.finish()


# -------------------------------------------------
# 🔟 CLI 파싱
# -------------------------------------------------
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Titanic – Lumo Homework 2")
    parser.add_argument(
        "--wandb",
        action=argparse.BooleanOptionalAction,
        default=False,
        help="Enable/disable Weights & Biases logging",
    )
    parser.add_argument(
        "-b",
        "--batch_size",
        type=int,
        default=32,
        help="Batch size for training (default: 32)",
    )
    parser.add_argument(
        "-e",
        "--epochs",
        type=int,
        default=2000,
        help="Number of training epochs (default: 2000)",
    )
    parser.add_argument(
        "-a",
        "--activation",
        type=str,
        default="relu",
        choices=["relu", "sigmoid", "elu", "leaky_relu"],
        help="Activation function to use in the network",
    )
    args = parser.parse_args()
    main(args)

🔧 BASE_PATH = /private/var/folders/8l/xmcsp2c54ngbycjxrnp58tm40000gn


FileNotFoundError: [Errno 2] No such file or directory: '/private/var/folders/8l/xmcsp2c54ngbycjxrnp58tm40000gn/_01_code/__init__.py'

# [요구사항 2] Activation Function 과 Batch Size 변경 및 선택하기

- 모델 구성 내에 Activation Function를 변경하여 더 나은 성능을 산출하는 Activation Function 이 무엇인지 조사하기
    - Sigmoid vs. ReLU vs. ELU vs. Leaky ReLU
- 모델 구성 내에 Training Batch Size를 변경하여 더 나은 성능을 산출하는 Batch Size 가 무엇인지 조사하기
    - 16, 32, 64, 128

모르겠다....

# [요구사항 3] 테스트 및 submission.csv 생성

1. 요구사항 2에서 살펴본 가장 좋은 성능을 보이는 Activation Function 및 Batch Size로 모델 구성하기
2. 사전에 테스트 데이터 (즉, test_data_loader) 구성하기
3. 훈련과정 중 어느 Epoch 시점에 테스트를 수행하여 submission.csv 를 구성해야 하는지 고찰하기
4. 고찰한 내용에 대한 추가 코딩 수행
5. submission.csv 생성하기

In [ ]:
모르겠다....

# [요구사항 4] submission.csv 제출 및 등수확인

1. Kaggle에 로그인 후 "Submit Prediction" 기능을 통한 submission.csv 제출
2. LeaderBoard에 등록된 나의 점수 및 위치 스크린 이미지 캡쳐하여 Jupyter Notebook에 넣기
    - 캡쳐 이미지를 클라우드에 업로드하여 해당 그림의 URL 생성필요

In [ ]:
모르겠다....

## 숙제 후기

#### 급하게 하느라 생각보다 힘들었다. 주피터 노트북으로 넣으니 경로 에러가 자꾸 발생했다.
#### 그냥 내기만이라도 하고 싶었는데 경로 부분부터 막혀서 실행도 되지 않았다.
#### 주피터 노트북에서 학습을 시켜야 하는지 파이썬 파일에서 시켜야 하는지도 감이 오지 않았다.
#### 수업 들을때와 집에서 복습할 때는 이해가 됬지만 코드를 타이타닉 코드에 맞게 수정할 때는 잘 안되었다. 아마 이해를 100퍼센트 못 한 거 같다.